# Do All 21 Spectral Bands Matter?
## Comparing AI Models and Reduced-Band Classification of Sea Ice and Leads

This notebook investigates binary classification of sea-ice/lead image patches from 21-band Earth-observation data.

**Research question:** Can sea ice and leads be classified accurately from 21-band satellite imagery using a computationally efficient machine-learning approach, and can spectral feature selection reduce model complexity without sacrificing classification performance?

The analysis compares Random Forest (RF), Convolutional Neural Network (CNN) and Vision Transformer (ViT) classifiers, applies the models to a spatial rollout region, evaluates Random Forest spectral-band importance, tests a five-band Random Forest, and compares computational and environmental cost.

## Data loading
The prepared NumPy arrays are stored in the project directory in Google Drive.


## Project workflow

The figure below summarises the complete analysis, from 21-band remote-sensing input data and IRIS reference labels through model comparison, spectral-band analysis, reduced-band testing and computational/environmental assessment.

![Workflow for Sea-Ice and Lead Classification](https://raw.githubusercontent.com/CalGorm/Sea-Ice-AI-Classification/main/figures/Workflow_Sea_Ice_Lead_Classification.png)

*Figure 1. Workflow for the sea-ice/lead classification project.*


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_path = '/content/drive/MyDrive/GEOL0069_Final_Project/'

import os
import numpy as np

X_train = np.load(os.path.join(save_path, 'X_train_balanced.npy'))
X_test = np.load(os.path.join(save_path, 'X_test_balanced.npy'))
y_train = np.load(os.path.join(save_path, 'y_train_balanced.npy'))
y_test = np.load(os.path.join(save_path, 'y_test_balanced.npy'))

## Convolutional Neural Network (CNN)

The CNN provides a compact deep-learning baseline for the 3 × 3 × 21 patches. A 2 × 2 convolution learns local spatial-spectral features, followed by max pooling, a 64-unit dense layer and a sigmoid output for binary classification. The model is trained for 10 epochs with Adam and binary cross-entropy.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Define CNN model
model = models.Sequential()
model.add(layers.Conv2D(32, (2, 2), activation='relu', input_shape=(3, 3, 21), padding='SAME'))
model.add(layers.MaxPooling2D((2, 2)))

# Dense classification layers
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))  # 1 neuron for binary classification


# Compile and train
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=10,
        validation_split=0.1)

Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


400/400 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7006 - loss: 0.5830 - val_accuracy: 0.7532 - val_loss: 0.4754
Epoch 2/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7279 - loss: 0.5529 - val_accuracy: 0.7623 - val_loss: 0.5524
Epoch 3/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7530 - loss: 0.5168 - val_accuracy: 0.7918 - val_loss: 0.5259
Epoch 4/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7723 - loss: 0.4866 - val_accuracy: 0.7574 - val_loss: 0.4666
Epoch 5/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7728 - loss: 0.4780 - val_accuracy: 0.7714 - val_loss: 0.4489
Epoch 6/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7714 - loss: 0.4835 - val_accuracy: 0.7679 - val_loss: 0.4588
Epoch 7/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7599 - loss: 0.4976 - val_accuracy: 0.7771 - val_loss: 0.5451
Epoch 8/10
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7754 - loss: 0.4751 - val_accuracy: 0.7532 - val_

In [ ]:
cnn_test_loss, cnn_test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"CNN test accuracy: {cnn_test_accuracy * 100:.2f}%")

CNN test accuracy: 78.48%


In [ ]:
model.save_weights('/content/drive/MyDrive/GEOL0069_Final_Project/CNN_sample1.weights.h5')

## Random Forest

Random Forest is used as the classical machine-learning baseline. Each 3 × 3 × 21 patch is flattened to 189 predictors and classified with an ensemble of 100 decision trees. This model is also used later for spectral feature-importance analysis and the 21-band versus five-band efficiency experiment.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialise Random Forest
clf = RandomForestClassifier(n_estimators=100)

# Flatten image patches for Random Forest
X_reshaped = np.reshape(X_train, (X_train.shape[0], -1))
# Fit the model
clf.fit(X_reshaped, y_train)

# Predict test classes
X_test_reshaped = np.reshape(X_test, (X_test.shape[0], -1))
y_pred = clf.predict(X_test_reshaped)

In [ ]:
from sklearn.metrics import accuracy_score

rf_test_accuracy = accuracy_score(y_test, y_pred)
print(f"Random Forest test accuracy: {rf_test_accuracy * 100:.2f}%")

Random Forest test accuracy: 93.42%


In [ ]:
import joblib

joblib.dump(
    clf,
    '/content/drive/MyDrive/GEOL0069_Final_Project/RandomForest_sample1.joblib'
)

['/content/drive/MyDrive/GEOL0069_Final_Project/RandomForest_sample1.joblib']

## Vision Transformer (ViT)

The ViT is the higher-complexity deep-learning comparison. The 3 × 3 × 21 input is resized before patch extraction and passed through stacked multi-head self-attention blocks. The classifier uses eight transformer layers, four attention heads and a two-class output. Training is run for 20 epochs with AdamW and the best validation weights are restored.


In [ ]:
# Install packages needed
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [ ]:
#=========================================================================================================
#=========================================================================================================
#=========================================================================================================

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super(Patches, self).__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches
#=========================================================================================================
#=========================================================================================================
#=========================================================================================================
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super(PatchEncoder, self).__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

#=========================================================================================================
#=========================================================================================================
#=========================================================================================================
def create_vit_classifier():
    inputs = layers.Input(shape=input_shape)
    # Augment data.
    augmented = more_data(inputs)
    # Create patches.
    patches = Patches(patch_size)(augmented)
    # Encode patches.
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Create multiple layers of the Transformer block.
    for _ in range(transformer_layers):
        # Layer normalization 1.
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        # Create a multi-head attention layer.
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        # Skip connection 1.
        x2 = layers.Add()([attention_output, encoded_patches])
        # Layer normalization 2.
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        # MLP.
        x3 = mlp(x3, hidden_units=transformer_units, dropout_rate=0.1)
        # Skip connection 2.
        encoded_patches = layers.Add()([x3, x2])

    # Create a [batch_size, projection_dim] tensor.
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.5)(representation)
    # Add MLP.
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    # Classify outputs.
    logits = layers.Dense(num_classes)(features)
    # Create the Keras model.
    model = keras.Model(inputs=inputs, outputs=logits)
    return model
#=========================================================================================================
#=========================================================================================================
#=========================================================================================================
def run_experiment(model):
    optimizer = keras.optimizers.AdamW(
        learning_rate=learning_rate, weight_decay=weight_decay
    )

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        ],
    )

    checkpoint_filepath = "/tmp/checkpoint.weights.h5"
    checkpoint_callback = keras.callbacks.ModelCheckpoint(
        checkpoint_filepath,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    )

    history = model.fit(
        x=X_train,
        y=y_train,
        batch_size=30,
        epochs=20,
        validation_split=0.1,
        callbacks=[checkpoint_callback],
    )

    model.load_weights(checkpoint_filepath)
    _, accuracy = model.evaluate(X_test, y_test)
    print(f"Test accuracy: {round(accuracy * 100, 2)}%")

    return history

In [ ]:
num_classes = 2 #Can be changed to multi-classed classification
input_shape = (3, 3, 21)#depends on the size of the image we want

learning_rate = 0.001
weight_decay = 0.0001
batch_size = 256
num_epochs = 100
image_size = 72
patch_size = 6
num_patches = (image_size // patch_size) ** 2
projection_dim = 64
num_heads = 4
transformer_units = [
    projection_dim * 2,
    projection_dim,
]
transformer_layers = 8
mlp_head_units = [2048, 1024]

In [ ]:
# Data augmentation
more_data = keras.Sequential(
    [
        layers.Normalization(),
        layers.Resizing(image_size, image_size),
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(factor=0.02),
        layers.RandomZoom(
            height_factor=0.2, width_factor=0.2
        ),
    ],
    name="more_data",
)
more_data.layers[0].adapt(X_train)

In [ ]:
vit = create_vit_classifier()
history = run_experiment(vit)

Epoch 1/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 65s 86ms/step - accuracy: 0.8350 - loss: 0.9911 - val_accuracy: 0.8544 - val_loss: 0.3752
Epoch 2/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 76s 82ms/step - accuracy: 0.8927 - loss: 0.2817 - val_accuracy: 0.8875 - val_loss: 0.2518
Epoch 3/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 35s 83ms/step - accuracy: 0.9054 - loss: 0.2283 - val_accuracy: 0.8889 - val_loss: 0.2539
Epoch 4/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 36s 84ms/step - accuracy: 0.9014 - loss: 0.2374 - val_accuracy: 0.9304 - val_loss: 0.1723
Epoch 5/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 40s 81ms/step - accuracy: 0.9096 - loss: 0.2130 - val_accuracy: 0.9170 - val_loss: 0.1956
Epoch 6/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 36s 85ms/step - accuracy: 0.9152 - loss: 0.2043 - val_accuracy: 0.9311 - val_loss: 0.1902
Epoch 7/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 36s 84ms/step - accuracy: 0.9155 - loss: 0.1947 - val_accuracy: 0.9318 - val_loss: 0.2347
Epoch 8/20
427/427 ━━━━━━━━━━━━━━━━━━━━ 34s 80ms/step - accuracy: 0.9211 - loss: 0.1959 - 

In [ ]:
vit.save_weights('/content/drive/MyDrive/GEOL0069_Final_Project/ViT_sample1.weights.h5')

## Model Comparison and Evaluation

All models are evaluated on the same held-out test set of 1,580 samples using accuracy, confusion matrices, precision, recall and F1 score.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np

# Random Forest predictions
rf_pred = clf.predict(X_test.reshape(X_test.shape[0], -1))

# CNN predictions
cnn_probs = model.predict(X_test, verbose=0).ravel()
cnn_pred = (cnn_probs >= 0.5).astype(int)

# ViT predictions
vit_logits = vit.predict(X_test, verbose=0)
vit_pred = np.argmax(vit_logits, axis=1)

for name, pred in [
    ("Random Forest", rf_pred),
    ("CNN", cnn_pred),
    ("ViT", vit_pred)
]:
    print("\n", name)
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, pred))
    print("Classification report:")
    print(classification_report(y_test, pred, digits=3))


 Random Forest
Accuracy: 0.9341772151898734
Confusion matrix:
[[734  41]
 [ 63 742]]
Classification report:
              precision    recall  f1-score   support

           0      0.921     0.947     0.934       775
           1      0.948     0.922     0.935       805

    accuracy                          0.934      1580
   macro avg      0.934     0.934     0.934      1580
weighted avg      0.935     0.934     0.934      1580


 CNN
Accuracy: 0.7848101265822784
Confusion matrix:
[[621 154]
 [186 619]]
Classification report:
              precision    recall  f1-score   support

           0      0.770     0.801     0.785       775
           1      0.801     0.769     0.785       805

    accuracy                          0.785      1580
   macro avg      0.785     0.785     0.785      1580
weighted avg      0.785     0.785     0.785      1580


 ViT
Accuracy: 0.9335443037974683
Confusion matrix:
[[723  52]
 [ 53 752]]
Classification report:
              precision    recall  f1-s

### Model-comparison result

RF achieved **93.42%** test accuracy (macro F1 **0.934**) and ViT achieved **93.35%** (macro F1 **0.934**), so their predictive performance was effectively tied. CNN accuracy was lower at **78.48%** (macro F1 **0.785**). This motivates comparing model complexity and computational cost rather than selecting the most complex model by default.


## Spatial Model Rollout

The trained classifiers are applied to the same 300 × 200 rollout region from `image2.npy`. Predictions are reconstructed from the inner 298 × 198 set of valid 3 × 3 patches to create spatial classification maps.


In [ ]:
rollout_image = np.load('/content/drive/MyDrive/GEOL0069_Final_Project/image2.npy')

x1, y1, x2, y2 = [100, 1000, 300, 1300]
rollout_roi = rollout_image[y1:y2, x1:x2]

print(rollout_roi.shape)

(300, 200, 21)


In [ ]:
rollout_patches = []

for i in range(1, rollout_roi.shape[0] - 1):
    for j in range(1, rollout_roi.shape[1] - 1):
        patch = rollout_roi[i-1:i+2, j-1:j+2, :]
        rollout_patches.append(patch)

rollout_patches = np.array(rollout_patches)

print(rollout_patches.shape)

(59004, 3, 3, 21)


In [ ]:
# Random Forest rollout prediction
rf_rollout_pred = clf.predict(
    rollout_patches.reshape(rollout_patches.shape[0], -1)
)

# Put predictions back into image shape
rf_rollout_inner = rf_rollout_pred.reshape(298, 198)

rf_rollout_image = np.zeros((300, 200))
rf_rollout_image[1:-1, 1:-1] = rf_rollout_inner

print(rf_rollout_image.shape)

(300, 200)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 9))
plt.imshow(rf_rollout_image, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.title('Random Forest Rollout')
plt.show()

<Figure size 600x900 with 1 Axes>

In [ ]:
plt.figure(figsize=(6, 9))
plt.imshow(rf_rollout_image, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.title('Random Forest Rollout')
plt.savefig('/content/drive/MyDrive/GEOL0069_Final_Project/RF_image2_sample1_25208373.png',
            bbox_inches='tight', dpi=300)
plt.show()

<Figure size 600x900 with 1 Axes>

In [ ]:
cnn_rollout_prob = model.predict(rollout_patches, verbose=0)
cnn_rollout_pred = (cnn_rollout_prob >= 0.5).astype(int)

cnn_rollout_inner = cnn_rollout_pred.reshape(298, 198)

cnn_rollout_image = np.zeros((300, 200))
cnn_rollout_image[1:-1, 1:-1] = cnn_rollout_inner

plt.figure(figsize=(6, 9))
plt.imshow(cnn_rollout_image, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.title('CNN Rollout')
plt.show()

<Figure size 600x900 with 1 Axes>

In [ ]:
plt.figure(figsize=(6, 9))
plt.imshow(cnn_rollout_image, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.title('CNN Rollout')

plt.savefig(
    '/content/drive/MyDrive/GEOL0069_Final_Project/CNN_image2_sample1_25208373.png',
    bbox_inches='tight',
    dpi=300
)

plt.show()

<Figure size 600x900 with 1 Axes>

In [ ]:
vit_rollout_logits = vit.predict(rollout_patches, verbose=0)
vit_rollout_pred = np.argmax(vit_rollout_logits, axis=1)

vit_rollout_inner = vit_rollout_pred.reshape(298, 198)

vit_rollout_image = np.zeros((300, 200))
vit_rollout_image[1:-1, 1:-1] = vit_rollout_inner

plt.figure(figsize=(6, 9))
plt.imshow(vit_rollout_image, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.title('ViT Rollout')
plt.show()

<Figure size 600x900 with 1 Axes>

In [ ]:
plt.figure(figsize=(6, 9))
plt.imshow(vit_rollout_image, cmap='gray', vmin=0, vmax=1)
plt.axis('off')
plt.title('ViT Rollout')

plt.savefig(
    '/content/drive/MyDrive/GEOL0069_Final_Project/ViT_image2_sample1_25208373.png',
    bbox_inches='tight',
    dpi=300
)

plt.show()

<Figure size 600x900 with 1 Axes>

## Spectral Band Importance and Reduced-Band Experiment

Random Forest impurity-based feature importance is reshaped to 3 × 3 × 21 and summed across the nine spatial positions to obtain one importance score per band. The five highest-ranked bands are then used in a reduced-input RF experiment.


In [ ]:
# Random Forest feature importance
rf_importance = clf.feature_importances_

# The RF used 3 x 3 pixels x 21 spectral bands
rf_importance_3d = rf_importance.reshape(3, 3, 21)

# Combine the 9 spatial positions to get one importance value per band
band_importance = rf_importance_3d.sum(axis=(0, 1))

print(band_importance)
print("Number of bands:", len(band_importance))

[0.19176189 0.14539486 0.0750634  0.05161178 0.02630312 0.03236476
 0.03420467 0.02492431 0.03101449 0.02539964 0.0240049  0.03033088
 0.02387357 0.01898152 0.04744259 0.03654449 0.0435873  0.03520245
 0.04509686 0.02625386 0.03063866]
Number of bands: 21


In [ ]:
import matplotlib.pyplot as plt

bands = np.arange(1, 22)

plt.figure(figsize=(10, 5))
plt.bar(bands, band_importance)
plt.xlabel('Spectral Band')
plt.ylabel('Random Forest Feature Importance')
plt.title('Spectral Band Importance for Sea-Ice/Lead Classification')
plt.xticks(bands)
plt.tight_layout()
plt.show()

# Print the five most important bands
ranked_bands = np.argsort(band_importance)[::-1]

print("Top 5 most important bands:")
for i in ranked_bands[:5]:
    print(f"Band {i + 1}: {band_importance[i]:.4f}")

<Figure size 1000x500 with 1 Axes>

Top 5 most important bands:
Band 1: 0.1918
Band 2: 0.1454
Band 3: 0.0751
Band 4: 0.0516
Band 15: 0.0474


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(bands, band_importance)
plt.xlabel('Spectral Band')
plt.ylabel('Random Forest Feature Importance')
plt.title('Spectral Band Importance for Sea-Ice/Lead Classification')
plt.xticks(bands)
plt.tight_layout()

plt.savefig(
    '/content/drive/MyDrive/GEOL0069_Final_Project/Spectral_Band_Importance.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

<Figure size 1000x500 with 1 Axes>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Top 5 bands identified above
top5_indices = ranked_bands[:5]

# ----- All 21 bands -----
X_train_21 = X_train.reshape(X_train.shape[0], -1)
X_test_21 = X_test.reshape(X_test.shape[0], -1)

rf_all21 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_all21.fit(X_train_21, y_train)
pred_21 = rf_all21.predict(X_test_21)
acc_21 = accuracy_score(y_test, pred_21)

# ----- Top 5 bands only -----
X_train_5 = X_train[:, :, :, top5_indices]
X_test_5 = X_test[:, :, :, top5_indices]

X_train_5 = X_train_5.reshape(X_train_5.shape[0], -1)
X_test_5 = X_test_5.reshape(X_test_5.shape[0], -1)

rf_top5 = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_top5.fit(X_train_5, y_train)
pred_5 = rf_top5.predict(X_test_5)
acc_5 = accuracy_score(y_test, pred_5)

print(f"21-band RF accuracy: {acc_21 * 100:.2f}%")
print(f"Top-5-band RF accuracy: {acc_5 * 100:.2f}%")
print(f"Accuracy difference: {(acc_5 - acc_21) * 100:.2f} percentage points")

21-band RF accuracy: 93.29%
Top-5-band RF accuracy: 93.99%
Accuracy difference: 0.70 percentage points


In [ ]:
seeds = [0, 10, 20, 30, 40]

acc_21_runs = []
acc_5_runs = []

for seed in seeds:
    rf21 = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    rf21.fit(X_train_21, y_train)
    acc_21_runs.append(
        accuracy_score(y_test, rf21.predict(X_test_21))
    )

    rf5 = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    rf5.fit(X_train_5, y_train)
    acc_5_runs.append(
        accuracy_score(y_test, rf5.predict(X_test_5))
    )

print("21-band mean accuracy:", np.mean(acc_21_runs) * 100)
print("5-band mean accuracy:", np.mean(acc_5_runs) * 100)
print("21-band standard deviation:", np.std(acc_21_runs) * 100)
print("5-band standard deviation:", np.std(acc_5_runs) * 100)

21-band mean accuracy: 93.0506329113924
5-band mean accuracy: 93.65822784810126
21-band standard deviation: 0.1088901932537047
5-band standard deviation: 0.2606488643287862


In [ ]:
labels = ['21 bands', 'Top 5 bands']

means = [
    np.mean(acc_21_runs) * 100,
    np.mean(acc_5_runs) * 100
]

stds = [
    np.std(acc_21_runs) * 100,
    np.std(acc_5_runs) * 100
]

plt.figure(figsize=(6, 5))
plt.bar(labels, means, yerr=stds, capsize=8)
plt.ylabel('Test Accuracy (%)')
plt.title('Random Forest Accuracy: 21 Bands vs Top 5 Bands')
plt.ylim(90, 95)
plt.tight_layout()
plt.show()

<Figure size 600x500 with 1 Axes>

In [ ]:
plt.figure(figsize=(6, 5))
plt.bar(labels, means, yerr=stds, capsize=8)
plt.ylabel('Test Accuracy (%)')
plt.title('Random Forest Accuracy: 21 Bands vs Top 5 Bands')
plt.ylim(90, 95)
plt.tight_layout()

plt.savefig(
    '/content/drive/MyDrive/GEOL0069_Final_Project/RF_21_vs_5_Bands.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

<Figure size 600x500 with 1 Axes>

In [ ]:
import time
import numpy as np

# Warm up the neural-network models first
model.predict(X_test[:100], verbose=0)
vit.predict(X_test[:100], verbose=0)

rf_times = []
cnn_times = []
vit_times = []

for _ in range(5):

    start = time.perf_counter()
    rf_all21.predict(X_test_21)
    rf_times.append(time.perf_counter() - start)

    start = time.perf_counter()
    model.predict(X_test, verbose=0)
    cnn_times.append(time.perf_counter() - start)

    start = time.perf_counter()
    vit.predict(X_test, verbose=0)
    vit_times.append(time.perf_counter() - start)

print(f"Random Forest mean prediction time: {np.mean(rf_times):.4f} seconds")
print(f"CNN mean prediction time: {np.mean(cnn_times):.4f} seconds")
print(f"ViT mean prediction time: {np.mean(vit_times):.4f} seconds")

Random Forest mean prediction time: 0.0828 seconds
CNN mean prediction time: 0.3778 seconds
ViT mean prediction time: 1.3279 seconds


In [ ]:
models = ['Random Forest', 'CNN', 'ViT']
accuracies = [93.42, 78.48, 93.35]
prediction_times = [
    np.mean(rf_times),
    np.mean(cnn_times),
    np.mean(vit_times)
]

plt.figure(figsize=(7, 5))

for name, x, y in zip(models, prediction_times, accuracies):
    plt.scatter(x, y, s=100)
    plt.annotate(name, (x, y), xytext=(6, 6),
                 textcoords='offset points')

plt.xlabel('Mean Prediction Time (seconds)')
plt.ylabel('Test Accuracy (%)')
plt.title('Model Accuracy vs Prediction Time')
plt.tight_layout()

plt.savefig(
    '/content/drive/MyDrive/GEOL0069_Final_Project/Accuracy_vs_Prediction_Time.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

<Figure size 700x500 with 1 Axes>

In [ ]:
import time

train_times_21 = []
train_times_5 = []

for seed in [0, 10, 20, 30, 40]:

    start = time.perf_counter()
    rf21_time = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    rf21_time.fit(X_train_21, y_train)
    train_times_21.append(time.perf_counter() - start)

    start = time.perf_counter()
    rf5_time = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    rf5_time.fit(X_train_5, y_train)
    train_times_5.append(time.perf_counter() - start)

print(f"21-band mean training time: {np.mean(train_times_21):.3f} seconds")
print(f"5-band mean training time: {np.mean(train_times_5):.3f} seconds")
print(f"Training-time reduction: {(1 - np.mean(train_times_5)/np.mean(train_times_21))*100:.1f}%")

21-band mean training time: 14.206 seconds
5-band mean training time: 5.716 seconds
Training-time reduction: 59.8%


### Computational-efficiency result

On the same Colab runtime, mean prediction times were **0.0828 s** for RF, **0.3778 s** for CNN and **1.3279 s** for ViT. For RF training, reducing the spectral input from 21 to five bands reduced mean training time from **14.206 s** to **5.716 s**, a **59.8% reduction**, while repeated-seed accuracy remained comparable.


## Environmental Cost Assessment

CodeCarbon is used to estimate energy consumption and associated CO2-equivalent emissions. Because a single RF fit is very short, the main environmental comparison repeats five complete RF training runs for the 21-band and five-band configurations. The repeated workload is used for the final relative comparison.


In [ ]:
!pip install -q codecarbon

from codecarbon import EmissionsTracker

print("CodeCarbon ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.9/396.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.8 MB/s eta 0:00:00
CodeCarbon ready


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from codecarbon import EmissionsTracker

project_dir = '/content/drive/MyDrive/GEOL0069_Final_Project'

tracker = EmissionsTracker(
    project_name='Data loading',
    output_dir=project_dir,
    output_file='emissions_data_loading.csv',
    measure_power_secs=1,
    log_level='error'
)

tracker.start()

X_train = np.load(os.path.join(project_dir, 'X_train_balanced.npy'))
X_test = np.load(os.path.join(project_dir, 'X_test_balanced.npy'))
y_train = np.load(os.path.join(project_dir, 'y_train_balanced.npy'))
y_test = np.load(os.path.join(project_dir, 'y_test_balanced.npy'))

tracker.stop()

results = pd.read_csv(
    os.path.join(project_dir, 'emissions_data_loading.csv')
).iloc[-1]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print(f"Energy used: {results['energy_consumed']:.8f} kWh")
print(f"Carbon emissions: {results['emissions']:.8f} kg CO2eq")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[codecarbon WARNING @ 23:50:04] Multiple instances of codecarbon are allowed to run at the same time.


X_train shape: (14212, 3, 3, 21)
X_test shape: (1580, 3, 3, 21)
Energy used: 0.00002704 kWh
Carbon emissions: 0.00001273 kg CO2eq


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Flatten 3x3x21 patches for Random Forest
X_train_21 = X_train.reshape(X_train.shape[0], -1)
X_test_21 = X_test.reshape(X_test.shape[0], -1)

tracker = EmissionsTracker(
    project_name='21-band Random Forest training',
    output_dir=project_dir,
    output_file='emissions_rf_21band.csv',
    measure_power_secs=1,
    log_level='error'
)

tracker.start()

rf_21_carbon = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_21_carbon.fit(X_train_21, y_train)

tracker.stop()

rf_21_pred = rf_21_carbon.predict(X_test_21)
rf_21_accuracy = accuracy_score(y_test, rf_21_pred)

results = pd.read_csv(
    os.path.join(project_dir, 'emissions_rf_21band.csv')
).iloc[-1]

print(f"Accuracy: {rf_21_accuracy * 100:.2f}%")
print(f"Energy used: {results['energy_consumed']:.8f} kWh")
print(f"Carbon emissions: {results['emissions']:.8f} kg CO2eq")

Accuracy: 93.29%
Energy used: 0.00009695 kWh
Carbon emissions: 0.00004564 kg CO2eq


In [ ]:
# Zero-based positions for Bands 1, 2, 3, 4 and 15
top5_indices = [0, 1, 2, 3, 14]

# Keep only those five bands
X_train_5 = X_train[:, :, :, top5_indices]
X_test_5 = X_test[:, :, :, top5_indices]

# Flatten for Random Forest
X_train_5 = X_train_5.reshape(X_train_5.shape[0], -1)
X_test_5 = X_test_5.reshape(X_test_5.shape[0], -1)

tracker = EmissionsTracker(
    project_name='5-band Random Forest training',
    output_dir=project_dir,
    output_file='emissions_rf_5band.csv',
    measure_power_secs=1,
    log_level='error'
)

tracker.start()

rf_5_carbon = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_5_carbon.fit(X_train_5, y_train)

tracker.stop()

rf_5_pred = rf_5_carbon.predict(X_test_5)
rf_5_accuracy = accuracy_score(y_test, rf_5_pred)

results = pd.read_csv(
    os.path.join(project_dir, 'emissions_rf_5band.csv')
).iloc[-1]

print(f"Accuracy: {rf_5_accuracy * 100:.2f}%")
print(f"Energy used: {results['energy_consumed']:.8f} kWh")
print(f"Carbon emissions: {results['emissions']:.8f} kg CO2eq")

Accuracy: 93.99%
Energy used: 0.00009777 kWh
Carbon emissions: 0.00004603 kg CO2eq


In [ ]:
seeds = [0, 10, 20, 30, 40]

# -----------------------------
# 21-band RF: 5 training runs
# -----------------------------
tracker21 = EmissionsTracker(
    project_name='21-band RF repeated training',
    output_dir=project_dir,
    output_file='emissions_rf_21band_repeated.csv',
    measure_power_secs=1,
    log_level='error'
)

tracker21.start()

for seed in seeds:
    rf_temp = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    rf_temp.fit(X_train_21, y_train)

tracker21.stop()

# -----------------------------
# 5-band RF: 5 training runs
# -----------------------------
tracker5 = EmissionsTracker(
    project_name='5-band RF repeated training',
    output_dir=project_dir,
    output_file='emissions_rf_5band_repeated.csv',
    measure_power_secs=1,
    log_level='error'
)

tracker5.start()

for seed in seeds:
    rf_temp = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1
    )
    rf_temp.fit(X_train_5, y_train)

tracker5.stop()

# Read results
r21 = pd.read_csv(
    os.path.join(project_dir, 'emissions_rf_21band_repeated.csv')
).iloc[-1]

r5 = pd.read_csv(
    os.path.join(project_dir, 'emissions_rf_5band_repeated.csv')
).iloc[-1]

print("21-band repeated training")
print(f"Energy: {r21['energy_consumed']:.8f} kWh")
print(f"Emissions: {r21['emissions']:.8f} kg CO2eq")

print("\n5-band repeated training")
print(f"Energy: {r5['energy_consumed']:.8f} kWh")
print(f"Emissions: {r5['emissions']:.8f} kg CO2eq")

print(
    f"\nEnergy difference: "
    f"{(1 - r5['energy_consumed']/r21['energy_consumed']) * 100:.1f}%"
)

print(
    f"Emissions difference: "
    f"{(1 - r5['emissions']/r21['emissions']) * 100:.1f}%"
)

21-band repeated training
Energy: 0.00044974 kWh
Emissions: 0.00021173 kg CO2eq

5-band repeated training
Energy: 0.00018756 kWh
Emissions: 0.00008830 kg CO2eq

Energy difference: 58.3%
Emissions difference: 58.3%


### Environmental-cost result

Across five RF training runs, the 21-band workload used **0.00044974 kWh** and emitted an estimated **0.00021173 kg CO2eq**. The five-band workload used **0.00018756 kWh** and emitted **0.00008830 kg CO2eq**, corresponding to a **58.3% reduction** in both measured energy and CodeCarbon-estimated emissions. Absolute values are very small and hardware-dependent, so the relative comparison is the more useful result.

## Limitations

- The training labels come from a limited manually labelled region, so results should not be assumed to generalise across locations, dates or sensors.
- Manual masks introduce possible labelling uncertainty.
- RF impurity-based feature importance can be affected by correlated predictors and does not prove physical wavelength importance.
- Five RF seeds provide a repeatability check rather than evidence of universal superiority.
- Prediction time and CodeCarbon estimates depend on the Colab hardware/runtime.
- The spatial rollout lacks independent ground truth across the full rollout region, so differences between model maps cannot automatically be labelled as errors.

## Conclusion

For this dataset, RF provided the strongest overall trade-off between accuracy, inference speed and interpretability. ViT matched RF accuracy but was much slower at inference, while CNN performed substantially worse. The reduced five-band RF preserved classification performance while cutting RF training time by about 60% and the repeated CodeCarbon workload by about 58%, showing that additional spectral and model complexity did not improve this specific classification task.
